In [1]:
# Import functions and constants from config.ipynb
%run phishing_detection_url_feature_extraction_functions.ipynb

import pandas as pd
import numpy as np
import os

#### Function to convert 'status' to integer

In [2]:
def map_status(status):
    """
    Maps a status string to an integer.

    Parameters:
    status (str): The status string to be mapped.

    Returns:
    int: 0 if status is "legitimate", 1 if status is "phishing".
         None if status is neither "legitimate" nor "phishing".
    """
    if status == "legitimate":
        return 0
    elif status == "phishing":
        return 1
    else:
        return None

#### Function to iterate through each URL, extract features, and append to df

In [3]:
def process_df(df):
    """
    Processes each row in the input DataFrame to extract features and append to a new DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame containing 'url' and 'status' columns.

    Returns:
    pd.DataFrame: DataFrame with 'url', 'status', and extracted features as columns.
    """
    output_rows = []
    for index, row in df.iterrows():
        url = row['url']
        status = map_status(row['status'])  # Map 'status' to integer
        features = extract_features(url)  # Extract features from URL
        output_row = {'url': url, 'status': status, **features}
        output_rows.append(output_row)
    return pd.DataFrame(output_rows)

In [4]:
def load_data_file(file_path):
    """
    Load data from a CSV file into a Pandas DataFrame.

    Parameters:
    file_path (str): The file path to the CSV file.

    Returns:
    pd.DataFrame: A Pandas DataFrame containing the loaded data.
    """
    try:
        # Load CSV file into DataFrame
        df = pd.read_csv(file_path)
        # Remove duplicates based on 'url' column, keeping the first occurrence
        cleaned_df = df.drop_duplicates(subset=['url'], keep='first')
        return cleaned_df
    except FileNotFoundError:
        raise FileNotFoundError(f"The file '{file_path}' does not exist. Please check the file path and try again.")
    except Exception as e:
        raise Exception(f"Error occurred while loading data from '{file_path}': {str(e)}")

In [5]:
def split_input_data(df):
    """
    Splits the input DataFrame into features (X) and target (y).

    Parameters:
    df (pd.DataFrame): The input DataFrame with features and target column.

    Returns:
    X (pd.DataFrame): DataFrame containing feature columns.
    y (pd.Series): Series containing the target column.
    """
    # Split the data into features and target
    X = df.drop('status', axis=1)
    y = df['status']
    return X, y

In [6]:
def print_df_size(df):
    rows, columns = df.shape
    print(f"Number of rows: {rows}")
    print(f"Number of columns: {columns}")

In [7]:
# Example usage (assuming the previous code snippet context):
if __name__ == "__main__":
    # Get the current working directory
    current_directory = os.getcwd()
    
    # Define the parent directory path
    parent_directory = os.path.abspath(os.path.join(current_directory, os.pardir))
    
    # Define the input file path relative to the parent directory
    input_file_path = os.path.join(parent_directory, 'data', 'data.csv')
    
    # Check if the input file exists
    if not os.path.isfile(input_file_path):
        raise FileNotFoundError(f"The file '{input_file_path}' does not exist. Please check the file path and try again.")
    
    try:
        # Load data from the CSV file and clean duplicates
        print(f"Loading data from {input_file_path}")
        input_df = load_data_file(input_file_path)
        
        #Add one phshing URL to make the balanced data set which is obTainfrom PhishTank
    
        new_data = {
            'url': 'https://timpqel.wixsite.com/my-site-1',
            'status': 'phishing'
        }

        # Create a DataFrame for the new row
        new_row_df = pd.DataFrame([new_data])

        # Ensure other columns are NaN
        for col in input_df.columns:
            if col not in new_row_df.columns:
                new_row_df[col] = np.nan

        # Concatenate the new row DataFrame with input_df
        input_df = pd.concat([input_df, new_row_df], ignore_index=True)
        
        # Define the output file path
        output_file_path = os.path.join(parent_directory, 'output', 'results', 'data_out.csv')
    
        # Process the input_df to extract features and append to output_df
        print("Processing data...")
        output_df = process_df(input_df)
        
        # Save the output_df to CSV
        print("Saving data...")
        output_df.to_csv(output_file_path, index=False)
    
        # Print confirmation message
        print(f"Output saved to: {output_file_path}")
        
        # Make a copy to use
        df = output_df.copy()
    
    except Exception as e:
        print(f"Error occurred: {str(e)}")


Loading data from /Users/gihansandakelum/Documents/phishing_url_detection/data/data.csv
Processing data...
Saving data...
Output saved to: /Users/gihansandakelum/Documents/phishing_url_detection/output/results/data_out.csv
